# GA4 JSON 派生模型审计

## tl;dr

- 完整公开 GA4 批次已独立展开为 3,982,732 个商品事件行和 14,821,470 个白名单参数行。
- 商品与参数模型均通过全部 BLOCKER：源数组对账、唯一键和展开上限/值类型检查。
- 商品身份缺失 130,641 行、白名单参数空值 2,080,202 行均保留为 WARNING；它们限制商品/参数解释，但不篡改 Raw JSON。


## Context & Methods

### Key Assumptions

本 Notebook 只审计 batch `55c6501e-5403-43a4-b4d5-1a422d87b9d5`、模型版本 `ga4_json_v1`。商品和参数数组独立展开，绝不互相 join，因此不会产生笛卡尔倍增。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from build_insight_evidence_package import connection_from_env

BATCH_ID = '55c6501e-5403-43a4-b4d5-1a422d87b9d5'
VERSION = 'ga4_json_v1'
connection = connection_from_env(PROJECT_ROOT / '.env')


## Data

### 1. Read derived-table counts


In [2]:
with connection.cursor() as cursor:
    cursor.execute('''
      SELECT
        (SELECT COUNT(*) FROM insightops_dw.ga4_item_event_stg
         WHERE batch_id = %s AND definition_version = %s) AS item_rows,
        (SELECT COUNT(*) FROM insightops_dw.ga4_event_param_stg
         WHERE batch_id = %s AND definition_version = %s) AS parameter_rows,
        (SELECT COUNT(DISTINCT item_id) FROM insightops_dw.ga4_item_event_stg
         WHERE batch_id = %s AND definition_version = %s
           AND item_id IS NOT NULL AND item_id <> '(not set)') AS distinct_item_ids
    ''', (BATCH_ID, VERSION, BATCH_ID, VERSION, BATCH_ID, VERSION))
    counts = cursor.fetchone()
counts


{'item_rows': 3982732, 'parameter_rows': 14821470, 'distinct_item_ids': 1393}

## Results

### 2. Review quality-rule outcomes


In [3]:
with connection.cursor() as cursor:
    cursor.execute('''
      SELECT run.dataset_name, result.rule_id, result.severity,
             result.failed_rows, result.outcome
      FROM insightops_ops.data_quality_run AS run
      JOIN insightops_ops.data_quality_result AS result ON result.run_id = run.run_id
      WHERE run.batch_id = %s AND run.rule_set_version = %s
        AND run.status = 'SUCCEEDED'
      ORDER BY run.started_at DESC, run.dataset_name, result.rule_id
      LIMIT 18
    ''', (BATCH_ID, VERSION))
    quality_rows = cursor.fetchall()
quality_rows


[{'dataset_name': 'ga4_event_param_stg',
  'rule_id': 'B01_allowlisted_params_match_source',
  'severity': 'BLOCKER',
  'failed_rows': 0,
  'outcome': 'PASSED'},
 {'dataset_name': 'ga4_event_param_stg',
  'rule_id': 'B02_parameter_key_unique',
  'severity': 'BLOCKER',
  'failed_rows': 0,
  'outcome': 'PASSED'},
 {'dataset_name': 'ga4_event_param_stg',
  'rule_id': 'B03_parameter_value_not_ambiguous',
  'severity': 'BLOCKER',
  'failed_rows': 0,
  'outcome': 'PASSED'},
 {'dataset_name': 'ga4_event_param_stg',
  'rule_id': 'W01_parameter_value_empty',
  'severity': 'WARNING',
  'failed_rows': 2080202,
  'outcome': 'FAILED'},
 {'dataset_name': 'ga4_event_param_stg',
  'rule_id': 'B01_allowlisted_params_match_source',
  'severity': 'BLOCKER',
  'failed_rows': 0,
  'outcome': 'PASSED'},
 {'dataset_name': 'ga4_event_param_stg',
  'rule_id': 'B02_parameter_key_unique',
  'severity': 'BLOCKER',
  'failed_rows': 0,
  'outcome': 'PASSED'},
 {'dataset_name': 'ga4_event_param_stg',
  'rule_id': 'B

## Takeaways

模型可作为后续商品、促销、支付与结账分析的受审计输入，但必须在查询中排除或显式处理 `(not set)` 商品身份及空参数值。商品和参数覆盖描述的是公开样例的技术质量，不可升级为生产业务结论。